##Chapter 6: Tools for Model Training and Experimenting/Project: 
* MultiClass classification for Computer Vision /Ch6-01-Training Base Model.py

In [0]:
!pip install -q pytorch-lightning==2.1.2 deltalake==0.14.0 deltatorch==0.0.3 evalidate==2.0.2 pillow==10.1.0 torchvision
dbutils.library.restartPython()

In [0]:
current_user = spark.sql("SELECT current_user()").first()[0]
volume_file_path = "/Volumes/porya_catalog/default/cv_clf"
train_delta_path = f"{volume_file_path}/train_imgs_main.delta"
valid_delta_path = f"{volume_file_path}/valid_imgs_main.delta"
volume_model_path = f"{volume_file_path}/intel_image_clf/models"

In [0]:
import torch
import torchvision
import pytorch_lightning as pl
from torch import nn, optim, Tensor, manual_seed, argmax
from torch.autograd import Variable
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel
from torch.optim import Optimizer
from torch.utils.data import TensorDataset, DataLoader
from torchmetrics.classification import Accuracy, MulticlassConfusionMatrix
from torchvision import transforms, models
from pytorch_lightning.utilities.model_summary import ModelSummary

import os
import numpy as np
import io
import logging
from math import ceil
from pyspark.sql.functions import col
from PIL import Image

train_df = (spark.read.format("delta")
            .load(train_delta_path)
            ).limit(10)

display(train_df)

try:
    display(dbutils.fs.ls(volume_model_path))
except Exception:
    print(f"Model directory not found at {volume_model_path} — it will be created during training.")

In [0]:
def mlflow_set_experiment(experiment_path:str = None):
    """
    Create or set your MLFlow experiment 
    experiment_path:str :: path where you want your experiment to be stored 
    """
    try:
        print(f"Setting our existing experiment {experiment_path}")
        mlflow.set_experiment(experiment_path)
        experiment = mlflow.get_experiment_by_name(experiment_path)
    except:
        print("Creating a new experiment and setting it")
        experiment = mlflow.create_experiment(name = experiment_path)
        mlflow.set_experiment(experiment_id=experiment_path)

In [0]:
import mlflow
# from mlia_utils import mlflow_funcs

experiment_path = f"/Users/{current_user}/intel-clf-training_action"
# mlflow_funcs.mlflow_set_experiment(experiment_path) 
mlflow_set_experiment(experiment_path) 
log_path = f"{volume_file_path}/intel_image_clf/intel_training_logger_board"
!mkdir {log_path}

In [0]:
class LitCVNet(pl.LightningModule):

        def __init__(self, num_classes = 6, learning_rate= 0.0001, family = "resnext", momentum = 0.1,  dropout_rate = 0):
            super().__init__()
            self.save_hyperparameters()
            self.family = family # we do not use familit explicitly, but you could play with 2 different family models using 1 script 
            self.momentum = momentum
            self.accuracy = Accuracy(task="multiclass", num_classes=num_classes)
            self.learning_rate = learning_rate 
            self.model = self.get_model(num_classes)
            self.loss = nn.CrossEntropyLoss()
            self.confusion_matrix = MulticlassConfusionMatrix(num_classes=num_classes)
            self.test_pred = []  # collect predictions

        def get_model(self, num_classes):
            """
            This is the function that initialises our model.
            If we wanted to use other prebuilt model libraries like timm we would put that model here
            """
            backbone = torchvision.models.wide_resnet50_2(pretrained=True)
            for param in backbone.parameters():
                param.required_grad = False
            num_ftrt = backbone.fc.in_features
            backbone.fc = nn.Linear(num_ftrt, num_classes)
            return backbone

        # We do not overwrite our forward pass 
        def forward(self, x):
            x  = self.model(x)
            return x
        
        def training_step(self, batch, batch_idx):
            x = batch["content"]
            y = batch["label_id"]
            logits = self.forward(x)
            loss = self.loss(logits,y)
            # Track accuracy
            acc = self.accuracy(logits, y)
            # required format
            self.log("loss", torch.tensor([loss]), on_step=True, on_epoch=True, logger=True)
            self.log("acc", torch.tensor([acc]), on_step=True, on_epoch=True, logger=True)
            return  {"loss": loss, "acc": acc}
        
        def validation_step(self, batch, batch_idx):
            x = batch["content"]
            y = batch["label_id"]
            logits = self.forward(x)
            loss = self.loss(logits, y)
            acc = self.accuracy(logits, y)
            # required format
            self.log("val_loss", torch.tensor([loss]), prog_bar=True, on_step=True, on_epoch=True, logger=True)
            self.log("val_acc", torch.tensor([acc]), prog_bar=True, on_step=True, on_epoch=True, logger=True)
            return {"val_loss": loss, "val_acc": acc} 
        
        def test_step(self, batch, batch_idx):
            x = batch["content"]
            y = batch["label_id"]
            # Evaluate model
            logits = self.forward(x)
            # Track loss
            loss = self.loss(logits, y)
            self.log('test_loss', loss)
            # Track accuracy
            acc = self.accuracy(logits, y)
            self.log('test_accuracy', acc)
            # Collect predictions
            self.test_pred.extend(logits.cpu().numpy())
            # Update confusion matrix
            self.confusion_matrix.update(logits, y)

        # predict_step is optional unless you are doing some predictions
        def predict_step(self, batch, batch_idx, dataloader_idx=0):
            x = batch["content"]
            y = batch["label_id"]
            preds = self.forward(x)
            return preds
        
        def configure_optimizers(self):
            params = self.model.fc.parameters()
            optimizer = torch.optim.AdamW(params, lr=self.learning_rate)
            lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[4,6], gamma=0.06)
            # required format 
            return {"optimizer":optimizer, "lr_scheduler":lr_scheduler}


# COMMAND ----------

model = LitCVNet()
ModelSummary(model, max_depth=4)

In [0]:
from deltatorch import create_pytorch_dataloader, FieldSpec

class DeltaDataModule(pl.LightningDataModule):
    """
    Creating a Data loading module with Delta Torch loader 
    """
    def __init__(self):
        super().__init__()
        self.num_classes = 6
        # Here we are applying the same Transformation 
        # in Production case scenario you probably would like to have to separate transformers for your data
        # 1 for Train and another one for test. 
        # See the Native Torch example below. 
        
        self.transform = transforms.Compose([
                transforms.Resize((150,150)),
                transforms.RandomHorizontalFlip(p=0.5), # randomly flip and rotate
                transforms.ColorJitter(0.3,0.4,0.4,0.2),
                transforms.ToTensor(),
                transforms.Normalize((0.425, 0.415, 0.405), (0.205, 0.205, 0.205))
                ])

    def dataloader(self, path: str, batch_size=32):
       
        return create_pytorch_dataloader(
            path,
            id_field="id",
            fields=[
                FieldSpec("content", load_image_using_pil=True, transform=self.transform),
                FieldSpec("label_id"),
            ],
            shuffle=True,
            batch_size=batch_size,
        )

    def train_dataloader(self):
        return self.dataloader(train_delta_path, batch_size=64)

    def val_dataloader(self):
        return self.dataloader(val_delta_path, batch_size=64)

    def test_dataloader(self):
        return self.dataloader(val_delta_path)


In [0]:
if not torch.cuda.is_available():
  print("⚠️ WARNING: No GPU found — running on CPU. Training will be significantly slower.")
else:
  print("✅ GPU is available.")

In [0]:
import mlflow

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
db_host = ctx.apiUrl().get()
db_token = ctx.apiToken().getOrElse(None)

# Alias to match variable name used in DeltaDataModule (Cell 8)
val_delta_path = valid_delta_path

MAX_EPOCH_COUNT = 2  # Reduced from 10 for CPU training on serverless
STEPS_PER_EPOCH = 5
EARLY_STOP_MIN_DELTA = 0.01
EARLY_STOP_PATIENCE = 10
STRATEGY = "auto"

from pytorch_lightning.loggers import TensorBoardLogger

def train_distributed(max_epochs: int = 1, strategy: str = "auto"):
    # import logging
    # logging.basicConfig(level=logging.DEBUG)

    # this is required if you want to log with MlFlow
    # if you do not have access to a Token, simply save models under a checkpoint and log them back
    # Track your loss with the Tensoflow Board 
    os.environ['DATABRICKS_HOST'] = db_host
    os.environ['DATABRICKS_TOKEN'] = db_token
    torch.set_float32_matmul_precision("medium")

    # Set up MLflow experiment and logger
    mlflow.set_experiment(f"/Users/{current_user}/intel-clf-training_action")
    mlflow_logger = pl.loggers.MLFlowLogger(experiment_name=f"/Users/{current_user}/intel-clf-training_action", tracking_uri="databricks")

    checkpoint_callback = pl.callbacks.ModelCheckpoint(
        save_top_k=2,
        mode="min",
        monitor="acc", # this has been saved under the Model Trainer - inside the validation_step function 
        dirpath=log_path,
        filename="sample-cvops-{epoch:02d}-{val_loss:.2f}"
    )
    early_stop_callback = pl.callbacks.EarlyStopping(
        monitor="loss",
        min_delta=EARLY_STOP_MIN_DELTA,
        patience=EARLY_STOP_PATIENCE,
        stopping_threshold=0.1,
        strict=False,
        verbose=True,
        mode="min",
        check_on_train_epoch_end=True,
        log_rank_zero_only=True
    )

    tqdm_callback = pl.callbacks.TQDMProgressBar(
        refresh_rate=STEPS_PER_EPOCH,
        process_position=0
    )
    # Initialize a trainer
    trainer = pl.Trainer(
        accelerator="auto",  # Changed from "gpu" to "auto" for serverless (CPU fallback)
        strategy=strategy,
        default_root_dir=log_path,
        max_epochs=max_epochs,
        logger=mlflow_logger,
        callbacks=[
            early_stop_callback,
            checkpoint_callback,
            tqdm_callback
        ],
    )

    print(f"Global Rank: {trainer.global_rank}")
    print(f"Local Rank: {trainer.local_rank}")
    print(f"World Size: {trainer.world_size}")

    dm = DeltaDataModule()
    model = LitCVNet(num_classes=6)
    #trainer.fit(model, dm)
    trainer.fit(model, train_dataloaders=dm.train_dataloader(), val_dataloaders=dm.val_dataloader())
    print("Training done!")

    # Log the trained model to MLflow so downstream cells can retrieve it
    reqs = mlflow.pytorch.get_default_pip_requirements() + [
        "pytorch-lightning==" + pl.__version__,
        "deltalake==0.14.0", "deltatorch==0.0.3"
    ]
    with mlflow.start_run(run_id=mlflow_logger.run_id):
        mlflow.pytorch.log_model(
            artifact_path="model",
            pytorch_model=model.model,
            pip_requirements=reqs,
        )
        mlflow.set_tag("ml2action", "cv_uc")
    print("Model logged to MLflow.")
    
    # Uncomment this if you are using DDP. 
    # AutoLog does not yet work well with the DDP. 
    # The best Practice with DDP would be to track your loss/acc with the Logger 
    # and then log your best_checkpoint back to the mlflow. 
    # In our case we use single node training so you will spot that the acc and loss were tracked. 
    # if strategy == "ddp":
    #     if trainer.global_rank == 0:
    #         # AutoLog does not work with DDP 
    #         from mlflow.models.signature import infer_signature
    #         with mlflow.start_run(run_name="running_cv_uc") as run:
                
    #             # Train the model ⚡🚅⚡
    #             print("We are logging our model")
    #             reqs = mlflow.pytorch.get_default_pip_requirements() + [
    #                 "pytorch-lightning==" + pl.__version__,
    #                 "deltalake==0.14.0","deltatorch==0.0.3"
    #             ]

    #             mlflow.pytorch.log_model(
    #                 artifact_path="model_cv_uc",
    #                 pytorch_model=model.model,
    #                 pip_requirements=reqs,
    #             )
    #             mlflow.set_tag("ml2action", "cv_uc")


# COMMAND ----------

# Disable deletion vectors on Delta tables (required for deltatorch compatibility)
for path in [train_delta_path, valid_delta_path]:
    spark.sql(f"ALTER TABLE delta.`{path}` SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'false')")
    spark.sql(f"REORG TABLE delta.`{path}` APPLY (PURGE)")

train_distributed(MAX_EPOCH_COUNT, STRATEGY)

# COMMAND ----------

# MAGIC %md 
# MAGIC To do the same but scaling vertically and horizontally if necessary use SparkTorchDistributor, just import the library and change the strategy mode for DDP or FSDP(if you are running GenAI models).
# MAGIC
# MAGIC Here is an example where we have run same code with 4 GPU on 1 single node, if you have multi node set, change the amount of nodes you have. 
# MAGIC ```
# MAGIC from pyspark.ml.torch.distributor import TorchDistributor
# MAGIC
# MAGIC distributed = TorchDistributor(num_processes=4, local_mode=True, use_gpu=True)
# MAGIC distributed.run(train_distributed, 1, "ddp")
# MAGIC ```
# MAGIC
# MAGIC Warning: this package works with 1 GPU per process, and it's in general not recommended to mix nthreads when you have more than 1 process.  
# MAGIC

# COMMAND ----------

# TorchDistributor requires a classic Spark cluster with GPUs — not supported on serverless
# from pyspark.ml.torch.distributor import TorchDistributor
# distributed = TorchDistributor(num_processes=4, local_mode=True, use_gpu=True)
# distributed.run(train_distributed, 1, "ddp")

##Chapter 6: Tools for Model Training and Experimenting/Project:
* MultiClass classification for Computer Vision /Ch6-02-Scoring Best Model.py

In [0]:

import os
import numpy as np
import io
import logging
from math import ceil

import torch
from torch import nn
from torch.autograd import Variable
from torch.nn import functional as F
from torch.nn.parallel import DistributedDataParallel
from torch.optim import Optimizer
from torch.utils.data import DataLoader
from torchvision import transforms, models
import pytorch_lightning as pl
from torchmetrics import Accuracy

from pyspark.sql.functions import col
from PIL import Image

In [0]:
def idx_class(df):            
    unique_object_ids = df.select("label_name").distinct().collect()
    object_id_to_class_mapping = {
        unique_object_ids[idx].label_name: idx for idx in range(len(unique_object_ids))}
    return object_id_to_class_mapping

In [0]:
train_df = (spark.read.format("delta").load(train_delta_path))
print(idx_class(train_df))


In [0]:

def select_best_model(experiment_path, artiffact_name = "model"):
  mlflow.set_experiment(experiment_path)
  best_model = mlflow.search_runs(
                filter_string=f'attributes.status = "FINISHED"',
                order_by=["metrics.acc DESC"],
                max_results=10,
                ).iloc[0]
  model_uri = "runs:/{0}/{1}".format(best_model.run_id, artiffact_name)
  local_path = mlflow.artifacts.download_artifacts(model_uri)
  return local_path 

In [0]:
import os
import mlflow
from mlflow.store.artifact.models_artifact_repo import ModelsArtifactRepository
# from mlia_utils.cv_clf_funcs import select_best_model

experiment_path = f"/Users/{current_user}/intel-clf-training_action"
local_path = select_best_model(experiment_path)

requirements_path = os.path.join(local_path, "requirements.txt")
if not os.path.exists(requirements_path):
  dbutils.fs.put("file:" + requirements_path, "", True)

device = "cuda" if torch.cuda.is_available() else "cpu"
loaded_model = torch.load(local_path+"/data/model.pth", map_location=torch.device(device), weights_only=False)
loaded_model

In [0]:
def transform_imgs(p=0.5):
  return torchvision.transforms.Compose([
      transforms.Resize((150,150)),
      transforms.RandomHorizontalFlip(p=p), # randomly flip and rotate
      transforms.ColorJitter(0.3,0.4,0.4,0.2),
      transforms.ToTensor(),
      transforms.Normalize((0.425, 0.415, 0.405), (0.205, 0.205, 0.205))
      ])

In [0]:
import matplotlib.pyplot as plt
from torch.autograd import Variable
# from mlia_utils import cv_clf_funcs as cv_funcs

# transform_tests = cv_funcs.transform_imgs()
transform_tests = transform_imgs()

# Get class labels and prediction image paths
outcomes = sorted([row.label_name for row in spark.read.format("delta").load(train_delta_path).select("label_name").distinct().collect()])
_pred_dir = f"{volume_file_path}/intel_image_clf/raw_images/seg_pred/seg_pred"
pred_files = [f.path.replace("dbfs:", "") for f in dbutils.fs.ls(_pred_dir)]


def pred_class(img):
    # transform images
    img_tens = transform_tests(img)
    # change image format (3,150,150) to (1,3,150,150) by help of unsqueeze function
    # image needs to be in cuda before predition
    img_im = img_tens.unsqueeze(0).to(device)
    uinput = Variable(img_im)
    uinput = uinput.to(device)
    out = loaded_model(uinput)
    # convert image to numpy format in cpu and snatching max prediction score class index
    index = out.data.cpu().numpy().argmax()    
    return index

classes = {k:v for k , v in enumerate(sorted(outcomes))}
loaded_model.eval()

plt.figure(figsize=(20,20))
for i, images in enumerate(pred_files[:50]):
    # just want 25 images to print
    if i > 24:break
    img = Image.open(images)
    index = pred_class(img)
    plt.subplot(5,5,i+1)
    plt.title(classes[index])
    plt.axis('off')
    plt.imshow(img)